# 🦆 DuckDB 快速入门（项目定制版）

本笔记本专为你的 `10_Fama_French_3` 项目准备，使用真实金融 CSV 数据练习 DuckDB。

**DuckDB 优势**：
- 直接对 CSV/Parquet 跑 SQL，无需先读 pandas
- 极快（远超 pandas 分析查询）
- 与 pandas / polars / Arrow 零拷贝互操作
- 纯 Python 包，已在 pyproject.toml 中声明

## 1. 连接与基本用法

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect()                    # 内存数据库（推荐）
print("DuckDB", duckdb.__version__)

## 2. 直接查询项目里的 CSV（最强功能）

In [ ]:
BASE = Path("10_Fama_French_3/data")
FACTORS = BASE / "F-F_Research_Data_Factors_daily.csv"

# 直接对 CSV 执行 SQL！
con.sql(f"SELECT * FROM read_csv_auto('{FACTORS}') LIMIT 5").df()

In [ ]:
# 查看结构
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{FACTORS}')").df()

## 3. 基础分析查询（Fama-French 因子）

In [ ]:
con.sql(f'''
    SELECT 
        strftime(TIME, '%Y-%m') AS ym,
        AVG("Mkt-RF") AS avg_mkt,
        AVG(SMB)      AS avg_smb,
        AVG(HML)      AS avg_hml
    FROM read_csv_auto('{FACTORS}')
    WHERE TIME >= '20200101'
    GROUP BY ym
    ORDER BY ym
    LIMIT 8
''').df()

## 4. 与 pandas 互操作（双向零拷贝）

In [ ]:
df = pd.read_csv(FACTORS, nrows=2000)

# DuckDB 可以直接查询 pandas DataFrame！
duckdb.sql('''
    SELECT TIME, "Mkt-RF" AS mkt 
    FROM df 
    WHERE "Mkt-RF" > 3 
    ORDER BY "Mkt-RF" DESC
''').df()

## 5. 多表 JOIN + 窗口函数（真实场景）

In [ ]:
PORT = BASE / "6_Portfolios_2x3_Daily.csv"

con.execute(f"CREATE OR REPLACE VIEW f AS SELECT * FROM read_csv_auto('{FACTORS}')")
con.execute(f"CREATE OR REPLACE VIEW p AS SELECT * FROM read_csv_auto('{PORT}')")

con.sql('''
    SELECT 
        strftime(f.TIME,'%Y-%m') AS ym,
        AVG(p."SMALL HiBM" - f.RF) AS small_hibm_excess,
        AVG(f."Mkt-RF")            AS mkt
    FROM f JOIN p ON f.TIME = p.TIME
    WHERE f.TIME >= '20200101'
    GROUP BY ym
    ORDER BY ym
    LIMIT 6
''').df()

## 6. 导出 Parquet（强烈推荐的分析格式）

In [ ]:
out_dir = Path("10_Fama_French_3/output")
out_dir.mkdir(exist_ok=True)
pq = out_dir / "factors.parquet"

con.sql(f'''
    COPY (
        SELECT TIME AS date, "Mkt-RF" AS mkt_rf, SMB, HML, RF
        FROM read_csv_auto('{FACTORS}')
    ) TO '{pq}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')
print("Parquet size:", round(pq.stat().st_size/1024,1), "KB")

## 7. 性能对比（感受 DuckDB 的速度）

In [ ]:
import time
def t(label, fn):
    s = time.perf_counter()
    r = fn()
    print(f"{label:22s} {time.perf_counter()-s:.3f}s  rows={len(r)}")

t("pandas",   lambda: pd.read_csv(FACTORS).query('`Mkt-RF` > 1.5'))
t("duckdb csv", lambda: con.sql(f"SELECT * FROM read_csv_auto('{FACTORS}') WHERE \"Mkt-RF\" > 1.5").df())
t("duckdb parquet", lambda: con.sql(f"SELECT * FROM read_parquet('{pq}') WHERE mkt_rf > 1.5").df())

## 8. 实用技巧 & 资源

- `con.register("name", df)` 把任意 DataFrame 变成 SQL 表
- `.df()`, `.arrow()`, `.polars()` 随意转换
- 官方中文/英文文档：https://duckdb.org/docs/
- 推荐搭配 Polars 使用（本项目已安装）

现在你可以把 Fama-French 代码逐步改用 DuckDB 加速！

In [ ]:
con.close()
print("学习完成！")